# LEAR FS3 Fine-Gamma Diagnostic

Scope:
- hourly
- D-only
- DA-only
- validation-only
- LEAR FS3 only
- no scenario or probability changes

This notebook reads a completed Phase D3 run folder. It does **not** rerun MILPs.


## Scope and reason

- test whether the CVaR response starts below gamma = 0.05
- identify whether the response is a true bid change or just a weak objective perturbation
- use validation weeks only
- keep scenarios, probabilities, and formulation fixed


In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display, Markdown, Image

RUN_DIR = Path(r'C:/Users/marijnvalk/PycharmProjects/Thesis/scripts/Data/03_Hydrogen_Test_Case/runs/20260519_135553_phase_d3_lear_fs3_fine_gamma_validation_diagnostic')
NOTEBOOK_INPUTS = RUN_DIR / 'notebook_inputs'
FIGURES_DIR = RUN_DIR / 'figures'
selected_weeks = pd.read_csv(NOTEBOOK_INPUTS / 'selected_weeks_manifest_table.csv')
support_days = pd.read_csv(NOTEBOOK_INPUTS / 'support_days.csv')
daily = pd.read_csv(NOTEBOOK_INPUTS / 'cvar_fine_gamma_daily.csv')
weekly = pd.read_csv(NOTEBOOK_INPUTS / 'cvar_fine_gamma_weekly.csv')
agg = pd.read_csv(NOTEBOOK_INPUTS / 'cvar_fine_gamma_aggregated.csv')
bid_diff = pd.read_csv(NOTEBOOK_INPUTS / 'bid_difference_by_gamma.csv')
tail_diag = pd.read_csv(NOTEBOOK_INPUTS / 'tail_scenario_diagnostics.csv')
daily_matrix = pd.read_csv(NOTEBOOK_INPUTS / 'daily_gamma_sensitivity_matrix.csv')
benchmark = pd.read_csv(NOTEBOOK_INPUTS / 'benchmark_metrics.csv')
perfect_foresight = pd.read_csv(NOTEBOOK_INPUTS / 'perfect_foresight_metrics.csv') if (NOTEBOOK_INPUTS / 'perfect_foresight_metrics.csv').exists() else pd.DataFrame()
validation_checks = pd.read_csv(NOTEBOOK_INPUTS / 'validation_checks_all_runs.csv')
cvar_checks = pd.read_csv(NOTEBOOK_INPUTS / 'cvar_validation_checks.csv')


## Validation weeks


In [ ]:
display(selected_weeks[['week_label', 'delivery_start_date', 'delivery_end_date', 'regime_label', 'selection_reason']])


## Aggregate gamma response


In [ ]:
display(agg.round(6))


## Daily gamma response


In [ ]:
display(daily_matrix.round(6))


## Bid-difference diagnostics


In [ ]:
display(bid_diff.round(9))


## Tail-scenario diagnostics


In [ ]:
display(tail_diag[['week_label', 'delivery_day', 'cvar_gamma', 'number_of_unique_scenario_profits', 'worst_scenario_profit', 'worst_scenario_probability_mass', 'cvar_tail_probability_mass', 'number_of_tail_scenarios', 'whether_tail_scenarios_change_vs_previous_gamma', 'whether_worst_scenario_changes_vs_previous_gamma']].round(6))


## Figures


In [ ]:
for figure_name in [
    'fig_fine_gamma_realised_profit.png',
    'fig_fine_gamma_cvar_tail_profit.png',
    'fig_fine_gamma_worst_scenario_profit.png',
    'fig_fine_gamma_clearing_ratio.png',
    'fig_fine_gamma_rejected_energy.png',
    'fig_fine_gamma_bid_difference_vs_gamma0.png',
    'fig_fine_gamma_shortfall.png',
    'fig_fine_gamma_risk_return_frontier.png',
    'fig_daily_gamma_heatmap_realised_profit.png',
    'fig_daily_gamma_heatmap_bid_difference.png',
]:
    path = FIGURES_DIR / figure_name
    if path.exists():
        display(Markdown(f'### {figure_name}'))
        display(Image(filename=str(path)))


## Benchmarks


In [ ]:
display(benchmark.round(6).head(30))


In [ ]:
display(perfect_foresight.round(6).head(30)) if not perfect_foresight.empty else display(Markdown('Perfect foresight not included.'))


## Conclusion prompt

Use the saved diagnostics in this order:
1. inspect `mean_total_abs_bid_quantity_difference_mwh_vs_gamma0` to see when bids actually move;
2. compare realised profit and CVaR tail profit at the same gamma;
3. check whether the tail scenario set itself changes across adjacent gamma values;
4. retain only gamma values that add information beyond the plateau.


## Validation status


In [ ]:
display(cvar_checks)


In [ ]:
display(validation_checks.groupby(['check_name', 'status'], as_index=False).size())
